# style-lora — Ukiyo-e ve Baroque

Sıfırdan yeniden kurulum. Önceki denemede üç şey aynı anda yanlıştı:

1. **Veri** — `huggan/wikiart`'ın Baroque'u 466 eser, **tek ressam** (Rembrandt), ve Ukiyo-e hiç yok.
2. **Tarif** — batch 1 ile 500 adım, yani modele 500 görsel gösterdik. Referans 8000 gösteriyor.
3. **Taban model** — sd-turbo yarım güç üstünde her şeyi tek noktaya çökertiyordu.

Üçü de değişti. Yeni çift **Ukiyo-e vs Baroque** çünkü iki şartı birden sağlıyor:
herkes bir bakışta ayırır, **ve** ikisi de insan/manzara/sahne resmediyor — yani
stiller *neyi* değil *nasıl* resmettiklerinde ayrılıyor. Barok karşı Soyut
Dışavurumculuk daha ayrık olurdu ve işe yaramazdı: tanınır nesneyi silen bir
adaptör yüksek puan alırdı, oysa ölçtüğümüz şey tam olarak stilin nesnelere
mal olduğu şey.

**Sıra:** veri → **bak** → kapılar → eğitim → **bak**. Ölçüm sonraki adımda.

## 1. GPU

In [ ]:
import subprocess

probe = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                       capture_output=True, text=True)
if probe.returncode != 0:
    raise SystemExit("No GPU. Runtime -> Change runtime type -> GPU, then run this cell again.")
print(probe.stdout.strip())

## 2. Kurulum

Colab torchao 0.10 ile geliyor ve peft 0.16'dan küçük bir sürüm bulunca hata
veriyor. Burada torchao'yu hiçbir şey kullanmıyor; kaldırmak yükseltmekten hızlı.
Kaldıktan sonra gerçekten gittiğini doğruluyoruz — `pip uninstall` sessizce
başarısız olabilir, `find_spec` olamaz.

In [ ]:
!git clone -q https://github.com/berkaykoklu/style-lora
%cd style-lora
!pip install -q -e .
!pip uninstall -y -q torchao

In [ ]:
import importlib.util

import peft
from diffusers.utils import logging as diffusers_logging

if importlib.util.find_spec("torchao") is not None:
    raise SystemExit("torchao is still installed; peft will refuse to build the LoRA layer")
print("peft", peft.__version__)

diffusers_logging.set_verbosity_error()

## 2b. Drive

Eğitim saatler sürüyor ve Colab kopunca sanal makineyle birlikte her şey gider.
Ağırlıklar ve hazırlanmış veri Drive'a yazılırsa kopma yalnızca o anki koşuya mal
olur; baştan çalıştırdığında bitmiş olanları atlar.

Telefondan çalıştıracaksan bu hücre isteğe bağlı değil — mobil tarayıcılar arka
plandaki sekmeyi uyutuyor ve oturum düşüyor.

In [ ]:
from pathlib import Path

USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/style-lora")
else:
    WORK = Path("/content/style-lora")

DATA = WORK / "data"
RUNS = WORK / "runs"
for folder in (DATA, RUNS):
    folder.mkdir(parents=True, exist_ok=True)
print("data ->", DATA)
print("runs ->", RUNS)

## 3. Veri

Stil başına 60 resim: ilk 40 eğitim havuzu, son 20 hiç eğitilmiyor. Ukiyo-e'de
toplam 66 eser var, sınırı o koyuyor.

Son 20 neden ayrı duruyor: stil merkezi ölçümün hedefi. Merkezi eğitim
resimlerinden kurarsak, bir tabloyu ezberleyen adaptör stili öğrenmeden hedefin
üstüne oturur.

In [ ]:
import pyarrow.parquet as pq

from stylelora.data import (
    PER_STYLE,
    STYLES,
    TRAIN_POOL,
    choose,
    download,
    split,
    style_index,
    write_images,
)

# The parquet stays on the VM: it is a gigabyte and re-downloads in a minute.
table = pq.read_table(download(Path("/content/paintings.parquet"))).to_pylist()
print(f"{len(table)} rows in the dataset")

for style in STYLES:
    folder = DATA / style
    if len(sorted(folder.glob("*.png"))) == PER_STYLE:
        print(f"  {style}: already on disk")
        continue
    matching = [r for r in table if r["labels"] == style_index(style)]
    saved = write_images(choose(matching, PER_STYLE), folder)
    print(f"  {style}: {len(matching)} available, {len(saved)} written")
    if len(saved) != PER_STYLE:
        raise SystemExit(f"{style} gave {len(saved)} of {PER_STYLE}")

for style in STYLES:
    pool, held = split(DATA / style)
    print(f"{style}: {len(pool)} train pool, {len(held)} held out")

## 4. Bak

Kapı iki setin **birbirinden** ayrıldığını sorabiliyor. Hiçbir zaman ikisinin
**etiketindeki stil olduğunu** soramıyor — ve geçen sefer üç saat, etiketini
taşımayan iki set üzerinde eğitim yaptık. Her sayı doğruydu, hepsi yanlış şey
hakkındaydı.

Bunu yakalayan tek şey bakmak.

In [ ]:
import matplotlib.pyplot as plt

from stylelora.data import contact_sheet

for style in STYLES:
    plt.figure(figsize=(11, 11 * 12 / 5))
    plt.imshow(contact_sheet(DATA / style))
    plt.title(style.replace("_", " "), fontsize=14)
    plt.axis("off")
    plt.show()

## 5. Caption'lar

Her resme konusunu söyleyen bir caption. Caption içeriği taşır, adaptöre stil
kalır — iki koşunun yalnızca stille ayrılması bunu gerektiriyor.

Caption modeli "an ukiyo-e print of a wave" yazabilir. O, stili metin
kodlayıcıya devreder ve adaptörü prompt'un zaten yaptığı bir iş üzerinden ölçer,
o yüzden stil kelimeleri yazılmadan önce ayıklanıyor.

In [ ]:
import torch
from transformers import BlipForConditionalGeneration, BlipProcessor

from stylelora.data import write_captions

_BLIP = "Salesforce/blip-image-captioning-base"
processor = BlipProcessor.from_pretrained(_BLIP)
captioner_model = BlipForConditionalGeneration.from_pretrained(_BLIP).to("cuda")


def blip(images):
    out = []
    for start in range(0, len(images), 8):
        batch = images[start : start + 8]
        inputs = processor(images=batch, return_tensors="pt").to("cuda")
        with torch.no_grad():
            ids = captioner_model.generate(**inputs, max_new_tokens=20)
        out.extend(processor.batch_decode(ids, skip_special_tokens=True))
    return out


for style in STYLES:
    folder = DATA / style
    written = write_captions(sorted(folder.glob("*.png")), folder, blip)
    print(f"\n{style}:")
    for name, text in list(written.items())[:5]:
        print(f"  {name}  {text}")

## 6. İki kapı

**Kelime kapısı.** CLIP bu iki stili kelimeyle ayırabiliyor mu? Bu, konudan
arınmış tek ölçüt — kelime resmin konusu hakkında hiçbir şey söylemiyor. Şans
%50. Eski çiftte bu %78 ile %92 arasındaydı.

**Ayrışma kapısı.** Görsel gömmelerde iki set ne kadar ayrık. Bu ölçüt konuyla
kirli, o yüzden tek başına karar verdirmiyor — ama düşük çıkarsa ölçümün dar bir
alanda çalışacağını söylüyor.

In [ ]:
from PIL import Image

from stylelora.score import embed_images, embed_text
from stylelora.separation import MARGIN_FLOOR, measure

held = {
    style: embed_images([Image.open(p) for p in split(DATA / style)[1]])
    for style in STYLES
}

WORDS = {"Ukiyo_e": "a japanese woodblock print", "Baroque": "a baroque oil painting"}
text = embed_text([WORDS[STYLES[0]], WORDS[STYLES[1]]])
right = sum(1 for v in held[STYLES[0]] if (v @ text[0]) > (v @ text[1]))
right += sum(1 for v in held[STYLES[1]] if (v @ text[1]) > (v @ text[0]))
total = len(held[STYLES[0]]) + len(held[STYLES[1]])
print(f"word gate:       {right}/{total} = {right / total:.0%}   (chance 50%)")

gate = measure(held[STYLES[0]], held[STYLES[1]])
print(f"within {STYLES[0]:10} {gate.within_a:.3f}")
print(f"within {STYLES[1]:10} {gate.within_b:.3f}")
print(f"between          {gate.between:.3f}")
print(f"margin           {gate.margin:.3f}   floor {MARGIN_FLOOR}   separated {gate.separated}")

if right / total < 0.7:
    raise SystemExit(
        "CLIP cannot sort these two by name; nothing measured later would mean anything"
    )

## 7. Eğitim

Referans tarifi: batch 4 × accumulation 2, 1000 adım, Min-SNR, gradient
clipping, yatay çevirme, ve batch'e göre ölçeklenen öğrenme oranı.

İkisi birebir aynı ayarlarla. Birini kurcalayıp diğerini bırakmak, tek farkı
olsun diye kurduğumuz karşılaştırmaya ikinci bir fark sokar.

Eğitim burada **Python fonksiyonu** olarak çağrılıyor, kabuktan değil: boru
hattının çıkış kodunu yutması bu projede üç kez başarısız bir kontrolün üstüne
commit attırdı. Fonksiyon hata verince hücre durur.

In [ ]:
import gc
import time

from stylelora.train import WEIGHTS_NAME, train


def run(style, rank=None, limit=None, steps=1000, seed=0):
    from stylelora.train import RANK

    rank = rank or RANK
    tag = f"n{limit or TRAIN_POOL}_r{rank}_t{steps}_s{seed}"
    out = RUNS / tag / style
    if (out / WEIGHTS_NAME).exists():
        print(f"  skip {tag}/{style}")
        return out / WEIGHTS_NAME
    started = time.perf_counter()
    train(style, images=DATA / style, out=out, steps=steps, seed=seed,
          rank=rank, limit=limit or TRAIN_POOL)
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  {tag}/{style} in {(time.perf_counter() - started) / 60:.1f} min")
    return out / WEIGHTS_NAME


weights = {style: run(style) for style in STYLES}

## 8. Bak (yine)

Aynı prompt, aynı tohum, üç sütun: taban model, ve iki adaptör tam güçte.

Sayıya geçmeden önce buraya bakmak gerekiyor. Sütunlar birbirinden ayırt
edilemiyorsa ölçecek bir şey yok, ve bunu 40 dakikalık bir süpürmeden sonra
öğrenmek istemeyiz.

In [ ]:
from stylelora.generate import PROMPTS, generate

SHOW = PROMPTS[:5]
columns = [("taban", generate(None, 0.0, SHOW, seed=0))]
for style in STYLES:
    columns.append((style.replace("_", " "), generate(weights[style], 1.0, SHOW, seed=0)))

fig, axes = plt.subplots(len(SHOW), len(columns), figsize=(4 * len(columns), 4 * len(SHOW)))
for c, (label, images) in enumerate(columns):
    for r, image in enumerate(images):
        axes[r][c].imshow(image)
        axes[r][c].set_xticks([])
        axes[r][c].set_yticks([])
        if r == 0:
            axes[r][c].set_title(label, fontsize=12)
for r, prompt in enumerate(SHOW):
    axes[r][0].set_ylabel(" ".join(prompt.split()[:4]), fontsize=8)
plt.tight_layout()
plt.show()

## Buraya kadar iyiyse

Bir sonraki adım ölçüm: güç süpürmesi, kontroller, ayrışma eğrisi ve körleme
test. Hepsi önceki kurulumda yazıldı ve çalışıyor; yalnızca bu adaptörlerin
üstünde yeniden koşturulacak.

Kötüyse burada duruyoruz — ölçülecek bir fark yoksa ölçmenin anlamı yok.